# Titanic Survivor Prediction

## Project Overview
The objective of this project is to predict whether a passenger survived the Titanic disaster using machine learning techniques. The project includes data preprocessing, feature engineering, model training, hyperparameter tuning, and model evaluation using a Random Forest Classifier.

## Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Loading the Dataset

In [ ]:
train_pd = pd.read_csv('data/train.csv')
test_pd = pd.read_csv('data/test.csv')

display(train_pd)

## Separating the Target Variable

The target variable (`Survived`) is extracted from the training dataset and stored separately. The remaining features will be used as inputs for the machine learning model.

In [ ]:
survived = train_pd['Survived']
train_pd = train_pd.drop(['Survived'], axis=1)

display(train_pd)

## Combining Training and Test Data

The training and test datasets are combined to ensure that preprocessing and feature engineering are applied consistently across both datasets.

In [ ]:
train_idx = train_pd['PassengerId']
test_idx = test_pd['PassengerId']

combined_pd = pd.concat([train_pd, test_pd]).reset_index(drop=True)

display(combined_pd)

## Identifying Missing Values

The dataset is inspected for missing values to determine which features require preprocessing before model training.

In [ ]:
print('Null Values: ')
print(combined_pd.isnull().sum())

## Exploring Age Distribution

The distribution of passenger ages is visualized to better understand the dataset and support the strategy for handling missing age values.

In [ ]:
y = combined_pd['Age'].value_counts()
x = y.index.values

plt.figure(figsize=(10, 5))
plt.bar(x, y)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

class_age = combined_pd[['Pclass', 'Age']]

for i in range(1, 4):
    plt.figure(figsize=(10, 5))
    plt.title('Class ' + str(i))
    y = class_age[class_age['Pclass'] == i]['Age'].value_counts()
    x = y.index.values

    plt.bar(x, y)
    plt.show()

medians = class_age.groupby('Pclass').median()
print(medians)

## Handling Missing Age Values

Missing age values are replaced using the median age of each passenger class (`Pclass`). This preserves class-specific age patterns and reduces the effect of outliers.

In [ ]:
for i in range(3):
    idx = np.where((combined_pd['Pclass'] == i + 1) & (combined_pd['Age'].isnull()))[0]
    combined_pd.loc[idx, 'Age'] = medians.values[i][0]

display(combined_pd)

## Handling Missing Fare Values

Missing fare values are replaced using the median fare of passengers with similar characteristics to maintain consistency within the dataset.

In [ ]:
display(combined_pd[combined_pd['Fare'].isnull()])

sim_fares = combined_pd[(combined_pd['Pclass'] == 3) & (combined_pd['Embarked'] == 'S')]['Fare']
print('Median Fare for Class 3 passengers who embarked at Southampton:', sim_fares.median())

plt.hist(sim_fares)
plt.plot()

combined_pd['Fare'] = combined_pd['Fare'].fillna(sim_fares.median())

## Handling Missing Embarked Values

Missing embarkation ports are filled using information from passengers with similar ticket class and fare values.

In [ ]:
display(combined_pd[combined_pd['Embarked'].isnull()])

sim_emb = combined_pd[(combined_pd['Pclass'] == 1) & (combined_pd['Fare'] >= 70) & (combined_pd['Fare'] <= 90)]['Embarked']
print(sim_emb.value_counts())

combined_pd['Embarked'] = combined_pd['Embarked'].fillna('C')

## Processing Cabin Information

Missing cabin values are replaced with the placeholder value `M`. Only the first character of each cabin value is retained to represent the passenger's deck.

In [ ]:
combined_pd['Cabin'] = combined_pd['Cabin'].fillna('M')
combined_pd['Cabin'] = combined_pd['Cabin'].str[0]

print(combined_pd['Cabin'].value_counts())

idx = np.where(combined_pd['Cabin'] == 'T')[0]
combined_pd.loc[idx, 'Cabin'] = 'M'

In [ ]:
display(combined_pd)

print(combined_pd.isnull().sum())

In [ ]:
print(combined_pd.nunique())

print(combined_pd['Pclass'].unique())

## Extracting Name-Based Features

Passenger names are decomposed into last names, titles, and first names. Titles such as Mr, Mrs, Miss, and Master may contain useful information related to survival patterns.

In [ ]:
print(combined_pd['Name'].unique())

names = combined_pd['Name']

last_names = []
titles = []
first_names = []

for name in names:
    if ', ' not in name:
        last_names.append('')
    else:
        last, name = name.split(', ', 1)
        last_names.append(last)

    if '. ' not in name:
        titles.append('')
    else:
        title, first = name.split('. ', 1)
        titles.append(title)
        first_names.append(first)

last_names = np.array(last_names)
titles = np.array(titles)
first_names = np.array(first_names)

idx = np.where(np.isin(titles, ['Capt', 'Col', 'Major']))
titles[idx] = 'Military'

idx = np.where(np.isin(titles, ['Don', 'Dona', 'Jonkheer', 'Lady', 'Sir', 'Master', 'the Countess']))
titles[idx] = 'Nobility'

idx = np.where(np.isin(titles, ['Miss', 'Mlle', 'Ms']))
titles[idx] = 'Ms'

idx = np.where(np.isin(titles, ['Mrs', 'Mme']))
titles[idx] = 'Mrs'

print('\n')
print(np.unique(titles, return_counts=True))

combined_pd['Titles'] = titles

display(combined_pd)


## Creating Age Groups

Passenger ages are grouped into discrete age ranges using `pd.cut()`. Binning reduces noise and helps capture age-related survival trends.

In [ ]:
print (combined_pd['Age'].min(), combined_pd['Age'].max())

bins = np.array([0,10,20,30,40,50,60,70,80])

combined_pd['Age_Bin'] = pd.cut(combined_pd['Age'], bins)

display(combined_pd)

## Calculating Ticket Frequency

The number of passengers sharing each ticket is calculated. This feature may indicate whether passengers were travelling alone or as part of a group.

In [ ]:
ticket_dict = dict(combined_pd['Ticket'].value_counts())

combined_pd['tkt_count'] = combined_pd['Ticket'].map(ticket_dict)

display(combined_pd)

## Creating Fare-Based Features

A new feature, `Fare_per_Ticket`, is created by dividing the ticket fare by the number of passengers sharing the ticket. The resulting values are grouped into fare ranges using binning.

In [ ]:
combined_pd['Fare_per_Ticket'] = combined_pd['Fare']/combined_pd['tkt_count']

plt.bar(combined_pd['Fare_per_Ticket'].value_counts().index.values, combined_pd['Fare_per_Ticket'].value_counts())
plt.show

print(combined_pd['Fare_per_Ticket'].min(), combined_pd['Fare_per_Ticket'].max())

bins = [0,20,40,60,80,150]
combined_pd['Fare_Bin'] = pd.cut(combined_pd['Fare_per_Ticket'], bins)

display(combined_pd)

## Creating Family Size Feature

Family size is calculated using the number of siblings/spouses (`SibSp`) and parents/children (`Parch`) accompanying each passenger.

Num_Family = SibSp + Parch + 1

In [ ]:
combined_pd['Num_Family'] = combined_pd['SibSp'] + combined_pd['Parch'] + 1

display(combined_pd)

## Removing Irrelevant Features

Columns that provide limited predictive value or contain highly unique values are removed to reduce noise and improve model performance.

In [ ]:
combined_pd = combined_pd.drop(
    columns=['Name', 'Age', 'Ticket', 'Fare', 'tkt_count', 'Fare_per_Ticket']
)

display(combined_pd)

In [ ]:
print(combined_pd.nunique())

In [ ]:
display(combined_pd)

In [ ]:
import sklearn

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler

In [ ]:
combined_pd = combined_pd.set_index('PassengerId')

In [ ]:
display(combined_pd)

## Label Encoding

Categorical features are converted into numerical values using Label Encoding so that they can be processed by machine learning algorithms.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = combined_pd.copy()
label_enc = label_enc.astype(str)

label_enc = label_enc.apply(LabelEncoder().fit_transform)

display(label_enc)

## One-Hot Encoding

Selected categorical variables are transformed using One-Hot Encoding to prevent the model from assuming ordinal relationships between categories.

In [ ]:
one_hot = label_enc.copy()

one_hot = pd.get_dummies(one_hot, columns=['Sex', 'Embarked', 'Titles'])
one_hot = one_hot.astype(int)

display(one_hot)

## Data Preparation and Train-Test Split

The processed dataset is scaled using Min-Max Scaling and divided into training and testing sets. The testing set is reserved for evaluating model performance on unseen data.

In [ ]:
x = one_hot.loc[train_idx].values
y = survived.values

scaler = MinMaxScaler()
scaler.fit(x)
x_scaled = scaler.fit_transform(x)

x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=0)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

## Training the Baseline Random Forest Model

A Random Forest Classifier is trained as the baseline model. Its performance serves as a reference before hyperparameter optimization.

In [ ]:
clf = RandomForestClassifier(random_state=0)
clf.fit(x_train, y_train).score(x_test, y_test)

## Hyperparameter Tuning with Randomized Search

RandomizedSearchCV is used to explore a wide range of Random Forest hyperparameters and identify promising parameter combinations.

In [ ]:
params = {'criterion':['gini','entropy'],
          'n_estimators' : [20,50,100,200,300,400,500,800,1000],
          'max_depth': np.arange(3,50),
          'min_samples_split': np.arange(1,1000),
          'max_features' : ['sqrt', 'log2'],
          'max_samples' : np.linspace(0.1,0.9,10)}

rand_search = RandomizedSearchCV(RandomForestClassifier(random_state=0), params, scoring='accuracy', random_state=0, cv=5)
rand_search.fit(x_train, y_train)

rand_params = rand_search.best_params_
print(rand_params, '\n')
print('Trian Acc:', rand_search.best_score_)
preds = rand_search.predict(x_test)
print('Test Acc:', accuracy_score(preds, y_test))

## Hyperparameter Refinement with Grid Search

GridSearchCV performs a more focused search around the best parameters identified by RandomizedSearchCV to further improve model performance.

In [ ]:
n_estimators = np.linspace(rand_params['n_estimators']-10, rand_params['n_estimators']+10, 3).astype(int)

min_samples_split = np.arange(rand_params['min_samples_split']-3, rand_params['min_samples_split']+3).astype(int)

max_samples =np.linspace(rand_params['max_samples']-.05, rand_params['max_samples']+.05, 6)

max_depth = np.arange(rand_params['max_depth']-5, rand_params['max_depth']+5).astype(int)


params = {'criterion': [rand_params['criterion']],
          'n_estimators' : n_estimators,
          'max_depth': max_depth,
          'min_samples_split': min_samples_split,
          'max_features' : [rand_params['max_features']],
          'max_samples' : max_samples}

grid_search = GridSearchCV(RandomForestClassifier(random_state=0), params, scoring='accuracy', cv=5)
grid_search.fit(x_train, y_train)

grid_params = grid_search.best_params_
print(grid_params, '\n')
print('Trian Acc:', grid_search.best_score_)
preds = grid_search.predict(x_test)
print('Test Acc:', accuracy_score(preds, y_test))

## Conclusion

A complete machine learning pipeline was developed for Titanic survival prediction. The project included missing value treatment, feature engineering, categorical encoding, scaling, model training, and hyperparameter optimization. The final Random Forest model achieved strong predictive performance and demonstrates the application of supervised machine learning techniques on tabular data.